In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

RANDOM_STATE = 42

DATA_DIR = Path(r"C:\DATA 4380\Tabular Project")
ARTIFACT_DIR = DATA_DIR / "model_artifacts"

print("Configuration ready.")
print("Artifact directory exists:", ARTIFACT_DIR.exists())

Configuration ready.
Artifact directory exists: True


In [3]:
def add_engineered_features(X): #add activity-volume, event-proportion, and diversity features
    X_new = X.copy()

    counts = X_new[BEHAVIORAL_FEATURES].astype(float)
    total = counts.sum(axis = 1)

    X_new["total_events"] = total #overall activity volume

    proportions = counts.div( #relative frequency of each event type
        total.replace(0, np.nan),
        axis=0,
    ).fillna(0)

    for col in BEHAVIORAL_FEATURES:
        X_new[f"{col}_proportion"] = proportions[col]

    X_new["event_type_diversity"] = (counts > 0).sum(axis=1) #number of event types with at least one occurrence

    log_proportions = np.log( #Shannon entropy of the event-type distribution
        proportions.where(proportions > 0, 1)
    )
    X_new["event_entropy"] = -(
        proportions * log_proportions
    ).sum(axis = 1)

    return X_new

def make_preprocessor(feature_columns, scale_numeric = False): #one-hot encode question number and optionally scale numeric features
    categorical = [
        col for col in CATEGORICAL_FEATURES
        if col in feature_columns
    ]
    numeric = [
        col for col in feature_columns
        if col not in categorical
    ]

    numeric_transformer = (
        StandardScaler() if scale_numeric else "passthrough"
    )

    return ColumnTransformer(
        transformers = [
            (
                "question",
                OneHotEncoder(handle_unknown = "ignore"),
                categorical,
            ),
            ("numeric", numeric_transformer, numeric),
        ],
        remainder = "drop",
    )

def make_models(feature_columns): #create baseline and candidate models for a given feature set
    return {
        "Majority class": DummyClassifier(
            strategy = "most_frequent"
        ),

        "Logistic Regression": Pipeline([
            (
                "preprocessor",
                make_preprocessor(
                    feature_columns,
                    scale_numeric = True,
                ),
            ),
            (
                "classifier",
                LogisticRegression(
                    max_iter = 1000,
                    random_state = RANDOM_STATE,
                ),
            ),
        ]),

        "Random Forest": Pipeline([
            (
                "preprocessor",
                make_preprocessor(feature_columns),
            ),
            (
                "classifier",
                RandomForestClassifier(
                    n_estimators = 200,
                    min_samples_leaf = 5,
                    random_state = RANDOM_STATE,
                    n_jobs = -1,
                ),
            ),
        ]),

        "LightGBM": Pipeline([
            (
                "preprocessor",
                make_preprocessor(feature_columns),
            ),
            (
                "classifier",
                lgb.LGBMClassifier(
                    objective = "binary",
                    n_estimators = 300,
                    learning_rate = 0.05,
                    num_leaves = 31,
                    random_state = RANDOM_STATE,
                    verbosity = -1,
                    n_jobs = -1,
                ),
            ),
        ]),

        "HistGradientBoosting": Pipeline([
            (
                "preprocessor",
                make_preprocessor(feature_columns),
            ),
            (
                "classifier",
                HistGradientBoostingClassifier(
                    max_iter = 200,
                    learning_rate = 0.1,
                    max_leaf_nodes = 31,
                    l2_regularization = 1.0,
                    random_state = RANDOM_STATE,
                ),
            ),
        ]),
    }

In [4]:
artifact_paths = {
    "Training data": ARTIFACT_DIR / "train_data.pkl",
    "Validation data": ARTIFACT_DIR / "val_data.pkl",
    "Test data": ARTIFACT_DIR / "test_data.pkl",
    "Event count columns": ARTIFACT_DIR / "event_count_columns.pkl",
}

missing_artifacts = [
    f"{name}: {path}"
    for name, path in artifact_paths.items()
    if not path.exists()
]

if missing_artifacts:
    raise FileNotFoundError(
        "Missing required artifacts:\n"
        + "\n".join(missing_artifacts)
    )

train_data = pd.read_pickle(artifact_paths["Training data"])
val_data = pd.read_pickle(artifact_paths["Validation data"])
test_data = pd.read_pickle(artifact_paths["Test data"])

event_count_columns = pd.read_pickle(
    artifact_paths["Event count columns"]
)

datasets = {
    "Train": train_data,
    "Validation": val_data,
    "Test": test_data,
}

print("All artifacts loaded successfully.")

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

print(f"Number of event-count features: {len(event_count_columns)}")

All artifacts loaded successfully.
Train: (296856, 22)
Validation: (63630, 22)
Test: (63630, 22)
Number of event-count features: 11


In [5]:
TARGET = "correct"

CATEGORICAL_FEATURES = ["question_number"]
BEHAVIORAL_FEATURES = event_count_columns.copy()

FEATURES = CATEGORICAL_FEATURES + BEHAVIORAL_FEATURES

print("Number of original features:", len(FEATURES))
print("Categorical features:", CATEGORICAL_FEATURES)
print("Behavioral features:", BEHAVIORAL_FEATURES)

Number of original features: 12
Categorical features: ['question_number']
Behavioral features: ['event_count_checkpoint', 'event_count_cutscene_click', 'event_count_map_click', 'event_count_map_hover', 'event_count_navigate_click', 'event_count_notebook_click', 'event_count_notification_click', 'event_count_object_click', 'event_count_object_hover', 'event_count_observation_click', 'event_count_person_click']


In [6]:
required_columns = FEATURES + [TARGET, "base_session_id"]

for name, df in datasets.items():
    missing_columns = set(required_columns) - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"{name} is missing required columns: {missing_columns}"
        )

    missing_values = df[required_columns].isna().sum().sum()

    print(f"\n{name}")
    print("Shape:", df.shape)
    print("Unique sessions:", df["base_session_id"].nunique())
    print("Missing values in required columns:", missing_values)
    print("Target distribution:")
    print(df[TARGET].value_counts(normalize=True).sort_index())

    if missing_values > 0:
        raise ValueError(
            f"{name} contains missing values in required columns."
        )


Train
Shape: (296856, 22)
Unique sessions: 16492
Missing values in required columns: 0
Target distribution:
correct
0    0.293311
1    0.706689
Name: proportion, dtype: float64

Validation
Shape: (63630, 22)
Unique sessions: 3535
Missing values in required columns: 0
Target distribution:
correct
0    0.295945
1    0.704055
Name: proportion, dtype: float64

Test
Shape: (63630, 22)
Unique sessions: 3535
Missing values in required columns: 0
Target distribution:
correct
0    0.297705
1    0.702295
Name: proportion, dtype: float64


In [7]:
train_sessions = set(train_data["base_session_id"])
val_sessions = set(val_data["base_session_id"])
test_sessions = set(test_data["base_session_id"])

overlaps = {
    "Train-validation": train_sessions & val_sessions,
    "Train-test": train_sessions & test_sessions,
    "Validation-test": val_sessions & test_sessions,
}

for name, overlap in overlaps.items():
    print(f"{name} overlap: {len(overlap)}")

assert all(len(overlap) == 0 for overlap in overlaps.values()), (
    "Session leakage detected between data splits."
)

print("All session splits are disjoint.")

Train-validation overlap: 0
Train-test overlap: 0
Validation-test overlap: 0
All session splits are disjoint.


In [8]:
for name, df in datasets.items():
    questions_per_session = (
        df.groupby("base_session_id")["question_number"].nunique()
    )
    rows_per_session = df.groupby("base_session_id").size()

    print(f"\n{name}")
    print("Unique question counts per session:")
    print(questions_per_session.value_counts().sort_index())
    print("Row counts per session:")
    print(rows_per_session.value_counts().sort_index())


Train
Unique question counts per session:
question_number
18    16492
Name: count, dtype: int64
Row counts per session:
18    16492
Name: count, dtype: int64

Validation
Unique question counts per session:
question_number
18    3535
Name: count, dtype: int64
Row counts per session:
18    3535
Name: count, dtype: int64

Test
Unique question counts per session:
question_number
18    3535
Name: count, dtype: int64
Row counts per session:
18    3535
Name: count, dtype: int64


In [9]:
X_train = train_data[FEATURES].copy()
y_train = train_data[TARGET].copy()

X_val = val_data[FEATURES].copy()
y_val = val_data[TARGET].copy()

X_test = test_data[FEATURES].copy()
y_test = test_data[TARGET].copy()

print("Feature matrix shapes:")
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).sort_index())

Feature matrix shapes:
Train: (296856, 12)
Validation: (63630, 12)
Test: (63630, 12)

Training target distribution:
correct
0    0.293311
1    0.706689
Name: proportion, dtype: float64


In [10]:
if "total_events_in_group" in train_data.columns:
    event_sum = train_data[BEHAVIORAL_FEATURES].sum(axis=1)
    stored_total = train_data["total_events_in_group"]

    matches = event_sum.eq(stored_total).sum()

    print(
        "Rows where event-count sum matches stored total:",
        f"{matches:,} / {len(train_data):,}"
    )

    display(event_sum.describe())
    display(X_train[BEHAVIORAL_FEATURES].describe().T)

Rows where event-count sum matches stored total: 296,856 / 296,856


count    296856.000000
mean        395.193286
std         197.856044
min          85.000000
25%         277.000000
50%         360.000000
75%         482.000000
max        4702.000000
dtype: float64

,count,mean,std,min,25%,50%,75%,max
event_count_checkpoint,296856.0,1.004891,0.071906,0.0,1.0,1.0,1.0,3.0
event_count_cutscene_click,296856.0,31.602777,25.207883,11.0,11.0,15.0,57.0,239.0
event_count_map_click,296856.0,8.234046,5.535443,1.0,6.0,7.0,9.0,170.0
event_count_map_hover,296856.0,14.717580,15.865215,0.0,5.0,11.0,19.0,381.0
event_count_navigate_click,296856.0,162.369051,123.952596,25.0,86.0,126.0,198.0,4030.0
event_count_notebook_click,296856.0,8.389182,11.165809,0.0,0.0,4.0,12.0,224.0
event_count_notification_click,296856.0,9.422286,2.711208,5.0,9.0,9.0,9.0,96.0
event_count_object_click,296856.0,37.401181,36.433336,6.0,19.0,26.0,42.0,939.0
event_count_object_hover,296856.0,18.167627,12.213377,0.0,11.0,17.0,24.0,146.0
event_count_observation_click,296856.0,3.084630,2.035916,0.0,2.0,3.0,4.0,53.0


In [11]:
X_train_engineered = add_engineered_features(X_train)
X_val_engineered = add_engineered_features(X_val)
X_test_engineered = add_engineered_features(X_test)

new_features = [
    col for col in X_train_engineered.columns
    if col not in X_train.columns
]

print("Original feature count:", X_train.shape[1])
print("Engineered feature count:", X_train_engineered.shape[1])
print("Number of added features:", len(new_features))

print("\nAdded features:")
print(new_features)

print("\nEngineered feature examples:")
display(
    X_train_engineered[
        [
            "question_number",
            "total_events",
            "event_type_diversity",
            "event_entropy",
        ]
    ].head()
)

Original feature count: 12
Engineered feature count: 26
Number of added features: 14

Added features:
['total_events', 'event_count_checkpoint_proportion', 'event_count_cutscene_click_proportion', 'event_count_map_click_proportion', 'event_count_map_hover_proportion', 'event_count_navigate_click_proportion', 'event_count_notebook_click_proportion', 'event_count_notification_click_proportion', 'event_count_object_click_proportion', 'event_count_object_hover_proportion', 'event_count_observation_click_proportion', 'event_count_person_click_proportion', 'event_type_diversity', 'event_entropy']

Engineered feature examples:


,question_number,total_events,event_type_diversity,event_entropy
0,1,165.0,10,1.601162
1,1,139.0,11,1.784748
5,1,154.0,11,1.881393
6,1,165.0,11,1.867515
8,1,146.0,9,1.586463


In [12]:
summary_features = [
    "total_events",
    "event_type_diversity",
    "event_entropy",
]

feature_sets = {
    "Original": FEATURES.copy(),
    "Activity summary": FEATURES + summary_features,
    "Full engineered": X_train_engineered.columns.tolist(),
}

for name, columns in feature_sets.items():
    print(f"{name}: {len(columns)} features")

Original: 12 features
Activity summary: 15 features
Full engineered: 26 features


In [13]:
for name, X in {
    "Train": X_train_engineered,
    "Validation": X_val_engineered,
    "Test": X_test_engineered,
}.items():
    missing = X.isna().sum().sum()
    infinite = np.isinf(
        X.select_dtypes(include = np.number)
    ).sum().sum()

    print(f"{name}: missing values={missing}, infinite values={infinite}")

    assert missing == 0, f"Missing values in {name} engineered features."
    assert infinite == 0, f"Infinite values in {name} engineered features."

assert (
    X_train_engineered.columns.tolist()
    == X_val_engineered.columns.tolist()
    == X_test_engineered.columns.tolist()
)

print("Engineered feature schemas match across all splits.")

Train: missing values=0, infinite values=0
Validation: missing values=0, infinite values=0
Test: missing values=0, infinite values=0
Engineered feature schemas match across all splits.


In [14]:
results = []
fitted_models = {}

for feature_set_name, feature_columns in feature_sets.items():
    print(f"\nTraining with feature set: {feature_set_name}")

    Xtr = X_train_engineered[feature_columns]
    Xv = X_val_engineered[feature_columns]

    models = make_models(feature_columns)

    for model_name, model in models.items():
        print(f"  Fitting {model_name}...")

        model.fit(Xtr, y_train)

        val_pred = model.predict(Xv)
        val_prob = model.predict_proba(Xv)[:, 1]

        results.append({
            "Feature set": feature_set_name,
            "Model": model_name,
            "Accuracy": accuracy_score(y_val, val_pred),
            "Precision": precision_score(y_val, val_pred, zero_division = 0),
            "Recall": recall_score(y_val, val_pred, zero_division = 0),
            "F1": f1_score(y_val, val_pred, zero_division = 0),
            "ROC-AUC": roc_auc_score(y_val, val_prob),
        })

        fitted_models[(feature_set_name, model_name)] = model

results_df = pd.DataFrame(results)

print("\nExperiment comparison complete.")
print("Number of experiments:", len(results_df))


Training with feature set: Original
  Fitting Majority class...
  Fitting Logistic Regression...
  Fitting Random Forest...
  Fitting LightGBM...


C:\Users\Tim Garen\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Tim Garen\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fitting HistGradientBoosting...

Training with feature set: Activity summary
  Fitting Majority class...
  Fitting Logistic Regression...
  Fitting Random Forest...
  Fitting LightGBM...


C:\Users\Tim Garen\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Tim Garen\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fitting HistGradientBoosting...

Training with feature set: Full engineered
  Fitting Majority class...
  Fitting Logistic Regression...
  Fitting Random Forest...
  Fitting LightGBM...


C:\Users\Tim Garen\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Tim Garen\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fitting HistGradientBoosting...

Experiment comparison complete.
Number of experiments: 15


In [15]:
leaderboard = results_df.sort_values( #ordered by ROC-AUC, descending
    "ROC-AUC",
    ascending = False
).reset_index(drop = True)

display(leaderboard.round(4))

,Feature set,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Activity summary,LightGBM,0.7492,0.7660,0.9268,0.8388,0.7654
1,Full engineered,LightGBM,0.7485,0.7654,0.9270,0.8385,0.7654
2,Original,LightGBM,0.7491,0.7664,0.9259,0.8386,0.7654
3,Original,HistGradientBoosting,0.7488,0.7661,0.9260,0.8385,0.7651
4,Full engineered,HistGradientBoosting,0.7489,0.7661,0.9261,0.8386,0.7650
5,Activity summary,HistGradientBoosting,0.7491,0.7668,0.9249,0.8385,0.7648
6,Original,Random Forest,0.7482,0.7646,0.9281,0.8385,0.7622
7,Activity summary,Random Forest,0.7474,0.7644,0.9269,0.8379,0.7612
8,Full engineered,Random Forest,0.7469,0.7638,0.9272,0.8376,0.7592
9,Full engineered,Logistic Regression,0.7463,0.7650,0.9231,0.8367,0.7583
